# Signature-Based Volatility Model Identification — walkthrough

Reproduces the core pipeline of **arXiv:2607.06340**: simulate Heston / Ornstein-Uhlenbeck / rough Bergomi volatility paths, compute truncated path signatures, and classify with XGBoost.

This notebook runs at a **small demo scale** (a few thousand paths, not the paper's 250,000/class) so it finishes in under a minute. See `README.md` for full-scale runs via `train.py`.

**Reproducibility note**: rough Bergomi paths here use an exact Cholesky fBM simulation rather than the paper's GPU hybrid scheme (mathematically equivalent at this step count, just not GPU-optimized); see the README's Reproducibility Notes for the full list of filled-in details.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt

from sig_vol_id.utils.config import Config, set_global_seed
from sig_vol_id.data.experiment_builder import ExperimentBuilder, EXPERIMENTS
from sig_vol_id.models.xgb_classifier import SignatureXGBClassifier
from sig_vol_id.evaluation.importance import ImportanceAnalyzer

cfg = Config.load('../configs/config.yaml')
set_global_seed(cfg.get('seed', default=42))
print('Available experiments:', list(EXPERIMENTS))

## 1. Build a small-scale version of Experiment 6.1 (Heston vs OU vs 2 rough Bergomi classes, random params)

In [ ]:
N_PATHS_DEMO = 3000   # paper uses 250,000
N_TEST_DEMO = 800     # paper uses 50,000

builder = ExperimentBuilder(cfg)
X_train, y_train, X_test, y_test, class_names = builder.build('6.1', N_PATHS_DEMO, N_TEST_DEMO, seed=0)
print(f"Classes: {class_names}")
print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

## 2. Train the XGBoost classifier (paper's exact hyperparameters)

In [ ]:
clf = SignatureXGBClassifier(
    n_classes=len(class_names),
    learning_rate=cfg['xgboost']['learning_rate'],
    max_depth=cfg['xgboost']['max_depth'],
    n_estimators=cfg['xgboost']['n_estimators'],
)
clf.fit(X_train, y_train)

train_acc = clf.accuracy(X_train, y_train)
test_acc = clf.accuracy(X_test, y_test)
print(f"Train accuracy: {train_acc:.4f}  (paper, full scale: ~1.00)")
print(f"Test accuracy:  {test_acc:.4f}  (paper, full scale: 0.9863)")

## 3. Confusion matrix (cf. paper's Figure 6.1)

In [ ]:
cm = clf.confusion_matrix(X_test, y_test, class_names, as_percentage=True)
print(cm)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm.values, cmap='Blues', vmin=0, vmax=100)
ax.set_xticks(range(len(class_names))); ax.set_xticklabels(class_names)
ax.set_yticks(range(len(class_names))); ax.set_yticklabels(class_names)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
for i in range(len(class_names)):
    for j in range(len(class_names)):
        ax.text(j, i, f"{cm.values[i,j]:.1f}", ha='center', va='center')
plt.title('Confusion matrix (%) -- demo scale, cf. paper Fig 6.1')
plt.colorbar(im)
plt.tight_layout()
plt.show()

## 4. Feature importance (cf. paper's Figure 6.4)

The paper finds that `sig_27` (an order-4 signature term) and `sig_13` (order-3) dominate importance, together accounting for ~50-60% of total importance.

In [ ]:
analyzer = ImportanceAnalyzer()
builtin = analyzer.builtin_importance(clf)
perm = analyzer.permutation_importance(clf, X_test, y_test, n_repeats=5)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
analyzer.truncate_at_cumulative(builtin, 0.9).plot.barh(ax=axes[0], title='Built-in Importance')
analyzer.truncate_at_cumulative(perm, 0.9).plot.barh(ax=axes[1], title='Permutation Importance')
axes[0].invert_yaxis(); axes[1].invert_yaxis()
plt.tight_layout()
plt.show()

print(f"Top built-in feature: {builtin.index[0]} ({builtin.iloc[0]:.3f})")
print(f"Top permutation feature: {perm.index[0]} ({perm.iloc[0]:.3f})")

## 5. The Heston/OU volatility-of-volatility effect (cf. paper's Section 6.8, Figure 6.11)

The paper's key mechanism finding: Heston misclassification as OU is driven by the volatility-of-volatility parameter `nu` -- low `nu` makes Heston geometrically indistinguishable from OU.

**Known open issue**: our current results for this specific experiment do NOT yet reproduce the paper's direction/magnitude reliably (see `README.md` Reproducibility Notes). The paper's footnote 7 says Heston's `nu` and OU's `sigma` are drawn from "comparable ranges" without giving the exact calibration, and getting this right enough to reproduce a 69.8%-at-low-nu / 9.1%-at-high-nu pattern needs further tuning of that comparability -- it is not simply a matter of using the same numeric range for both, since Heston's multiplicative sqrt(v)-scaled noise and OU's additive noise respond differently to a given diffusion-coefficient value. This is flagged as an open item rather than glossed over.

In [ ]:
# Quick demo-scale check: low-nu vs high-nu Heston/OU confusion.
# NOTE: as of this notebook, this does not yet reliably reproduce the paper's
# direction (low nu -> MORE confusion). Treat this cell as a diagnostic, not
# a verified reproduction -- see the markdown above and README.md.
_, _, X_low, y_low, names_low = builder.build('6.8_low_nu', N_PATHS_DEMO, N_TEST_DEMO, seed=1)
clf_low = SignatureXGBClassifier(n_classes=len(names_low), n_estimators=200, max_depth=6)
clf_low.fit(*builder.build('6.8_low_nu', N_PATHS_DEMO, N_TEST_DEMO, seed=1)[:2])

_, _, X_high, y_high, names_high = builder.build('6.8_high_nu', N_PATHS_DEMO, N_TEST_DEMO, seed=2)
clf_high = SignatureXGBClassifier(n_classes=len(names_high), n_estimators=200, max_depth=6)
clf_high.fit(*builder.build('6.8_high_nu', N_PATHS_DEMO, N_TEST_DEMO, seed=2)[:2])

cm_low = clf_low.confusion_matrix(X_low, y_low, names_low, as_percentage=True)
cm_high = clf_high.confusion_matrix(X_high, y_high, names_high, as_percentage=True)

heston_idx = names_low.index('Heston')
ou_idx = names_low.index('OU')
print(f"Low nu:  Heston misclassified as OU: {cm_low.iloc[heston_idx, ou_idx]:.1f}%  (paper: 69.8%)")
print(f"High nu: Heston misclassified as OU: {cm_high.iloc[heston_idx, ou_idx]:.1f}%  (paper: 9.1%)")
print("\nIf low-nu shows LOWER confusion than high-nu here, that's the known open issue --\n"
      "see README.md Reproducibility Notes, item on Section 6.8 nu-comparability.")

## Next steps

- Full-scale run: `python train.py --config ../configs/config.yaml --experiment 6.1` (250,000 paths/class)
- All 9 named experiments: see `README.md`'s Quickstart section
- Figure 6.4 reproduction at full scale: `python run_feature_importance.py --results-dir results/6.2`